# Leaguepedia data pipeline and quality checks

Run the pipeline cells in order. Each export is a separate cell, and fetch cells make read-only Leaguepedia requests before replacing their corresponding CSV. They require the existing `mwrogue` setup and credentials configured for `AuthCredentials(user_file="me")`.

The audit cells are read-only and can also be run by themselves against the current CSVs. Repeated records and uncertain top-50 player matches are reported for review, not automatically deleted or guessed.

In [1]:
from pathlib import Path
from collections import Counter
import csv
import os
import subprocess
import sys

SEARCH_ROOT = next(
    (
        path for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "fetchLeaguepedia.py").is_file()
        or (path / "fetch-data" / "fetchLeaguepedia.py").is_file()
    ),
    None,
)
if SEARCH_ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the esport_data project.")

ROOT = SEARCH_ROOT / "fetch-data" if (SEARCH_ROOT / "fetch-data" / "fetchLeaguepedia.py").is_file() else SEARCH_ROOT
PROJECT_ROOT = ROOT.parent
OUTPUT_DIR = (ROOT / "../output").resolve()
os.environ["ESPORT_OUTPUT_DIR"] = str(OUTPUT_DIR)

OUTPUT_FILES = {
    "fetchLeaguepedia.py": "leagues.csv",
    "fetchTournament.py": "tournaments.csv",
    "fetchTeam.py": "teams.csv",
    "fetchPlayer.py": "players.csv",
    "fetchChampion.py": "champions.csv",
    "fetchTeamResult.py": "team_results.csv",
    "fetchRoster.py": "roster.csv",
    "fetchPlayerChampStats.py": "player_champ_stats.csv",
}

def run_export(script):
    output_path = OUTPUT_DIR / OUTPUT_FILES[script]
    print(f"Running {script}; output: {output_path}")
    subprocess.run(
        [sys.executable, "-u", str(ROOT / script), "--out", str(output_path)],
        cwd=PROJECT_ROOT,
        check=True,
    )

def read_csv(name):
    path = OUTPUT_DIR / name
    with path.open(newline="", encoding="utf-8-sig") as csv_file:
        reader = csv.DictReader(csv_file)
        return reader.fieldnames or [], list(reader)

## 1. Fetch leagues

In [2]:
run_export("fetchLeaguepedia.py")

Running fetchLeaguepedia.py; output: D:\Works\esport_data\output\leagues.csv


## 2. Fetch tournaments

In [3]:
run_export("fetchTournament.py")

Running fetchTournament.py; output: D:\Works\esport_data\output\tournaments.csv


## 3. Fetch teams

In [5]:
run_export("fetchTeam.py")

Running fetchTeam.py; output: D:\Works\esport_data\output\teams.csv


## 4. Fetch players

In [6]:
run_export("fetchPlayer.py")

Running fetchPlayer.py; output: D:\Works\esport_data\output\players.csv


## 5. Fetch champions

In [7]:
run_export("fetchChampion.py")

Running fetchChampion.py; output: D:\Works\esport_data\output\champions.csv


## 6. Fetch team results

In [10]:
run_export("fetchTeamResult.py")

Running fetchTeamResult.py; output: D:\Works\esport_data\output\team_results.csv


## 7. Fetch roster

In [11]:
run_export("fetchRoster.py")

Running fetchRoster.py; output: D:\Works\esport_data\output\roster.csv


## 8. Fetch player champion statistics

Scoreboard players are matched by Leaguepedia link, falling back to name when the link is empty. Restart this export from batch 1 to regenerate earlier batches after changing player matching.

In [ ]:
run_export("fetchPlayerChampStats.py")

Running fetchPlayerChampStats.py; output: D:\Works\esport_data\output\player_champ_stats.csv
